In [15]:
import sys
sys.path.append('..')

import pandas as pd
from utils.db_utils import write_table

In [16]:
df = pd.read_csv("../../data/Total Graduates by Age Group.csv")
df.head(10)

,statistics,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,≤ 24_grad_degree,150.3,156.8,176.0,195.5,140.1,136.6,150.0,170.1,186.7
1,25 - 34_grad_degree,975.8,1087.9,1129.3,1185.2,1033.8,1112.0,1072.3,1101.9,1134.1
2,35 - 44_grad_degree,568.4,610.6,686.4,759.3,817.2,888.5,971.0,1035.7,1107.7
3,≥ 45_grad_degree,480.1,529.2,575.8,635.8,641.8,676.4,778.1,815.1,852.7
4,≤ 24_grad_diploma,478.4,503.1,506.7,531.9,432.2,447.8,448.6,471.0,486.1
5,25 - 34_grad_diploma,871.5,880.4,943.7,965.7,865.6,883.2,930.9,898.0,906.7
6,35 - 44_grad_diploma,408.8,453.9,524.4,553.1,603.9,639.6,669.3,729.6,764.8
7,≥ 45_grad_diploma,348.9,379.8,401.5,460.3,452.7,464.5,491.9,521.8,542.5


In [17]:
def transform_grad_by_age(file_path):
    df = pd.read_csv(file_path)

    # 1. Pivot dataframe
    df_long = df.melt(
        id_vars=["statistics"],
        var_name="year",
        value_name="value"
    )

    # 2. Parse statistics into age_group and qualification
    def parse_stats(stat_name):
        if "_" in stat_name:
            age_part, qual_part = stat_name.split("_", 1)
            return age_part.strip(), qual_part.strip()
        return "Total", stat_name.strip()

    df_long[['age_group', 'qual_cat']] = df_long['statistics'].apply(
        lambda x: pd.Series(parse_stats(x))
    )

    # 3. Extract qualification from qual_cat
    def extract_qualification(qual_cat):
        if pd.isna(qual_cat):
            return None
        if "_" in qual_cat:
            return qual_cat.split("_", 1)[1]
        return qual_cat

    df_long['qualification'] = df_long['qual_cat'].apply(extract_qualification)

    # 4. Convert numeric and scale
    df_long['total_graduate'] = pd.to_numeric(df_long['value'], errors='coerce')
    df_long = df_long.dropna(subset=['total_graduate'])
    df_long['total_graduate'] = df_long['total_graduate'] * 1000

    df_final = df_long[['year', 'age_group', 'qualification', 'total_graduate']].copy()
    df_final = df_final.sort_values(['year', 'age_group', 'qualification']).reset_index(drop=True)

    return df_final

In [18]:
df_final = transform_grad_by_age("../../data/Total Graduates by Age Group.csv")
df_final.head(10)

,year,age_group,qualification,total_graduate
0,2016,25 - 34,degree,975800.0
1,2016,25 - 34,diploma,871500.0
2,2016,35 - 44,degree,568400.0
3,2016,35 - 44,diploma,408800.0
4,2016,≤ 24,diploma,478400.0
5,2016,≤ 24,degree,150300.0
6,2016,≥ 45,degree,480100.0
7,2016,≥ 45,diploma,348900.0
8,2017,25 - 34,degree,1087900.0
9,2017,25 - 34,diploma,880400.0


In [19]:
write_table(df_final, "sc_bronze", "dosm_graduates_age")

Table sc_bronze.dosm_graduates_age written successfully.
